In [1]:
#imports and proxies
import json
import pyodbc
from simple_salesforce import Salesforce
import pandas as pd
import numpy as np
pd.set_option("display.max_colwidth", None)

#set proxies
proxies = {
 	"http": 'http://web.prod.proxy.cargill.com:4200',
    "https":'http://web.prod.proxy.cargill.com:4200',
} 

#set salesforce instance: leap_prod, leap_sandbox, salt_prod, salt_sandbox, avenger_prod, avenger_sandbox
#gets credentials from JSON file for the selected SF instance
def sf_grabber(instance, proxies):
    with open("creds.json") as f:
        json_load = json.load(f)['instances']
    
    #returns creds for specific instance
    creds = next((item for item in json_load if item['name'] == instance), None)
    
    #connect and query
    sf_kwargs  = {
        'username': creds['username'],
        'password': creds['pass'],
        'organizationId': creds['org_id'],
        'proxies': proxies
    }
    
    #ads test domain to arguement if "sanbox" / adds token if available for avenger
    if "sandbox" in instance:
        sf_kwargs["domain"] = "test"
    if creds.get("token"):
        sf_kwargs["security_token"] = creds["token"]
    return Salesforce(**sf_kwargs)

#steablish SF connections
salt_sf = sf_grabber('salt_prod', proxies)
leap_sf = sf_grabber('leap_prod', proxies)

In [9]:
#get salt cases
#VALUES HERE FEEL WRONG SHOULD INVESTIGATE
salt_cases_df = salt_sf.query_all("""
SELECT Id,
QN_Number__c,  
Account.Name,
Sales_order_number__c,
CS_Case_Number__c,
Additional_Information__c,
Case_Owner__c,
ClosedDate
FROM Case
WHERE CS_BusinessUnit__c = 'Salt'
AND Reason = 'Complaint'
AND Sales_order_number__c = ''
AND QN_Number__c = ''
""")
salt_cases_df = pd.json_normalize(salt_cases_df["records"])[[
    'Id',
    'QN_Number__c',
    'Sales_order_number__c',
    'CS_Case_Number__c',
    'Additional_Information__c',
    'Case_Owner__c',
    'ClosedDate',
    'Account.Name'
]]
salt_cases_df['CS_Case_Number__c'] = salt_cases_df['CS_Case_Number__c'].astype(int)

#string operation for salt
salt_cases_df["summary_text"] = salt_cases_df.apply(
    lambda row: (
        f"{row['QN_Number__c']} - "
        f"{row['Account.Name']} on order "
        f"{row['Sales_order_number__c']}, has a(n) "
        f"{row['Additional_Information__c']}. The case number is "
        f"{row['CS_Case_Number__c']}. Processed by  "
        f"{row['Case_Owner__c']} "
        f"{pd.Timestamp.today().strftime('%m/%d/%Y')}"
    ),
    axis=1
)

print(len(salt_cases_df))
salt_cases_df.sample(n = 10)
#salt_cases_df[salt_cases_df['CS_Case_Number__c'] == 1707841]

2565


,Id,QN_Number__c,Sales_order_number__c,CS_Case_Number__c,Additional_Information__c,Case_Owner__c,ClosedDate,Account.Name,summary_text
1902,5005x00001sIx4YAAS,None,None,1285809,None,Cynthia Cline,2023-11-17T16:23:47.000+0000,LEPRINO FOODS - WEST,"None - LEPRINO FOODS - WEST on order None, has a(n) None. The case number is 1285809. Processed by Cynthia Cline 04/30/2026"
2226,500UI000000WHfRYAW,None,None,1292062,None,Michael Harris,2023-11-07T17:17:48.000+0000,CAPITOL FOOD COMPANY,"None - CAPITOL FOOD COMPANY on order None, has a(n) None. The case number is 1292062. Processed by Michael Harris 04/30/2026"
322,5005x00001qbUTBAA2,None,None,1216443,None,Jan Klusener,2023-08-03T16:24:32.000+0000,COSTCO PEWAUKEE WI 1101,"None - COSTCO PEWAUKEE WI 1101 on order None, has a(n) None. The case number is 1216443. Processed by Jan Klusener 04/30/2026"
1800,5005x00001sHr8lAAC,None,None,1281289,None,Jamie Leonard,2023-10-24T14:55:43.000+0000,FOOD INGREDIENTS INC WI,"None - FOOD INGREDIENTS INC WI on order None, has a(n) None. The case number is 1281289. Processed by Jamie Leonard 04/30/2026"
2231,500UI000000WjV3YAK,None,None,1292214,None,Michael Harris,2023-11-09T16:28:26.000+0000,SUN CHEMICAL CORP CINCINNATI,"None - SUN CHEMICAL CORP CINCINNATI on order None, has a(n) None. The case number is 1292214. Processed by Michael Harris 04/30/2026"
1029,5005x00001rXLKcAAO,None,None,1244026,None,Katrina Lofton,2023-09-12T15:15:09.000+0000,MCKEE FOODS CORPORATION,"None - MCKEE FOODS CORPORATION on order None, has a(n) None. The case number is 1244026. Processed by Katrina Lofton 04/30/2026"
2073,5005x00001sX9SKAA0,None,None,1274104,None,Tyler Scott,2023-10-26T14:09:49.000+0000,BATORY WAREHOUSE MASON,"None - BATORY WAREHOUSE MASON on order None, has a(n) None. The case number is 1274104. Processed by Tyler Scott 04/30/2026"
1435,5005x00001s2p3tAAA,None,None,1251828,None,Missy Donaldson,2023-09-13T19:04:40.000+0000,FOSTER POULTRY FARMS- AP DEHLI,"None - FOSTER POULTRY FARMS- AP DEHLI on order None, has a(n) None. The case number is 1251828. Processed by Missy Donaldson 04/30/2026"
1788,5005x00001sHnK1AAK,None,None,1280754,None,Saundra Reimer,2023-10-23T21:29:14.000+0000,PROTENERGY NATURAL FOODS CAMBRIDGE,"None - PROTENERGY NATURAL FOODS CAMBRIDGE on order None, has a(n) None. The case number is 1280754. Processed by Saundra Reimer 04/30/2026"
2157,5005x00001sY5YVAA0,None,None,1277325,None,Carol Crawford,2023-10-17T21:22:43.000+0000,MISSION FOODS RANCHO CUCAMONGA,"None - MISSION FOODS RANCHO CUCAMONGA on order None, has a(n) None. The case number is 1277325. Processed by Carol Crawford 04/30/2026"


In [3]:
#get leap cases
#VALUES HERE FEEL WRONG SHOULD INVESTIGATE
leap_cases_df = leap_sf.query_all("""
SELECT Id,
Account.Name,
Sales_Order_Number__c,
Product__r.Name,
CaseNumber,
CaseUpdate__c,
Owner.Name,
ClosedDate,
credit_debit_memo_number__c
FROM Case
WHERE Quality_Notification_ID__c  <> ''
AND Status = 'Closed'
""")
leap_cases_df = pd.json_normalize(leap_cases_df["records"])[[
    'Id',
    'Account.Name',
    'Sales_Order_Number__c',
    'Product__r.Name',
    'CaseNumber',
    'CaseUpdate__c',
    'Owner.Name',
    'ClosedDate',
    'credit_debit_memo_number__c'
]]

#string operation for leap
leap_cases_df["summary_text"] = leap_cases_df.apply(
    lambda row: (
        f"{row['Account.Name']} on order  "
        f"{row['Sales_Order_Number__c']} has a complaint case "
        f"{row['CaseNumber']} for "
        f"{row['CaseUpdate__c']} "
        f"{row['Product__r.Name']}. "
        f"{row['credit_debit_memo_number__c']}"
    ),
    axis=1
)

print(len(leap_cases_df))
leap_cases_df

4


,Id,Account.Name,Sales_Order_Number__c,Product__r.Name,CaseNumber,CaseUpdate__c,Owner.Name,ClosedDate,credit_debit_memo_number__c,summary_text
0,5002M00000kLqs6QAC,TOA KASEI,None,12748 COLD SWL HYD ST PH 25KG 70025 A39S,00030608,None,Jumpei Nagano,2025-07-18T00:57:15.000+0000,None,TOA KASEI on order None has a complaint case 00030608 for None 12748 COLD SWL HYD ST PH 25KG 70025 A39S. None
1,5002M00000kMxmRQAS,TOA KASEI,None,12748 COLD SWL HYD ST PH 25KG 70025 A39S,00030725,None,Jumpei Nagano,2025-07-18T00:57:26.000+0000,None,TOA KASEI on order None has a complaint case 00030725 for None 12748 COLD SWL HYD ST PH 25KG 70025 A39S. None
2,5002M00000nraIIQAY,JOHNSON & JOHNSON,None,NaN,00032472,None,Septa Lestari,2025-05-27T06:18:39.000+0000,None,JOHNSON & JOHNSON on order None has a complaint case 00032472 for None nan. None
3,5002M000014Jw86QAC,CONAGRA BRANDS,4065772,100001364 - PROSANTE TVGPTN 10B MNCD 50LB BG,00041764,None,Gwen Cuddeback,2025-10-06T18:06:45.000+0000,None,CONAGRA BRANDS on order 4065772 has a complaint case 00041764 for None 100001364 - PROSANTE TVGPTN 10B MNCD 50LB BG. None
